# JobShield · PhoBERT v1 trên Kaggle
Gắn dataset riêng tư từ gói train-val, bật GPU và Internet. Chọn Save Version → Save & Run All. Khi hoàn tất, tải jobshield-phobert-v1-result.zip ở Output.

In [ ]:
import os, shutil, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Đang kiểm tra GPU và Java...", flush=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Session options trước khi chạy.")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
subprocess.run(["java", "-version"], check=True)
print("Đang cài thư viện...", flush=True)
%pip install "transformers==4.57.6" "accelerate==1.12.0" "py-vncorenlp==0.1.4"

In [ ]:
import hashlib, json, tempfile, zipfile
from pathlib import Path

input_dir = Path("/kaggle/input")
root = Path(tempfile.mkdtemp(prefix="jobshield-"))
expected_hashes = {'train.jsonl': '5923016ec0f91748c8d3fc823398d36083252342805f461681579b46f3150c10', 'validation.jsonl': '42618d308b2e600918bd81a47aaf65559e3a2b7503ac44ae0b923cfaad180ddf'}
def load_splits(folder):
    # Kaggle thường giải nén ZIP khi tạo dataset. Chấp nhận cả ZIP nguyên bản.
    candidates = [p for p in folder.rglob("train.jsonl") if (p.parent / "validation.jsonl").is_file()]
    for path in candidates + list(folder.rglob("jobshield-v1-train-val.zip")):
        if path.suffix == ".zip":
            with zipfile.ZipFile(path) as archive:
                if not set(expected_hashes).issubset(archive.namelist()):
                    continue
                blobs = {n: archive.read(n) for n in expected_hashes}
        else:
            blobs = {n: (path.parent / n).read_bytes() for n in expected_hashes}
        if all(hashlib.sha256(buf).hexdigest() == expected_hashes[n] for n, buf in blobs.items()):
            return [[json.loads(line) for line in blobs[n].splitlines()] for n in ("train.jsonl", "validation.jsonl")]
    raise ValueError("Chưa tìm thấy đúng dữ liệu JobShield. Dùng Add Input để gắn dataset tạo từ gói train-val.")

train, val = load_splits(input_dir)
train = [r for r in train if r["overall_label"] is not None]
assert (len(train), len(val)) == (585, 41)
codes = sorted({f["code"] for r in train for f in r["output"]["findings"]})
print(len(train), "mẫu train có nhãn,", len(val), "mẫu validation. Mã học được:", codes)

In [ ]:
def encode_document(sentences, segmenter, tokenizer):
    # Giữ câu đang xét ở giữa. Với mẫu dài, ưu tiên phần ngữ cảnh gần câu đó.
    parts = []
    for sentence in sentences:
        text = " ".join(segmenter.word_segment(sentence["text"]))
        prefix = sentence["type"] + " " + sentence["speaker"] + ": "
        parts.append(tokenizer.encode(prefix + text, add_special_tokens=False, verbose=False))
    sep = [tokenizer.sep_token_id]
    stream, starts = [], []
    for part in parts:
        starts.append(len(stream))
        stream.extend(part + sep)
    features = []
    for start, part in zip(starts, parts):
        focus = part[:252]
        available = 252 - len(focus)
        after = start + len(part) + 1
        left = min(start, available // 2)
        right = min(len(stream) - after, available - left)
        left = min(start, available - right)
        tokens = stream[start-left:start] + sep + focus + sep + stream[after:after+right]
        features.append({"input_ids": tokenizer.build_inputs_with_special_tokens(tokens)})
    return features

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import py_vncorenlp
from scipy.special import expit
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

set_seed(42)
model_id = "vinai/phobert-base"
segmenter_dir = root / "vncorenlp"
segmenter_dir.mkdir(exist_ok=True)
print("Đang tải VnCoreNLP...", flush=True)
py_vncorenlp.download_model(save_dir=str(segmenter_dir))
segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(segmenter_dir))
print("Đang tải tokenizer PhoBERT...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

def prepare(rows, name):
    features, keys = [], []
    for row in tqdm(rows, desc=name):
        sentences = row["input"]["sentences"]
        gold = {(f["code"], sid) for f in row["output"]["findings"] for sid in f["evidence_sentence_ids"]}
        for s, item in zip(sentences, encode_document(sentences, segmenter, tokenizer), strict=True):
            item["labels"] = [float((code, s["id"]) in gold) for code in codes]
            features.append(item)
            keys.append((row["sample_id"], s["id"]))
    return features, keys

train_features, _ = prepare(train, "Tách câu train")
val_features, keys = prepare(val, "Tách câu validation")
counts = np.sum([x["labels"] for x in train_features], axis=0)
weights = torch.tensor(np.sqrt((len(train_features) - counts) / counts), dtype=torch.float32)

In [ ]:
print("Đang nạp trọng số PhoBERT...", flush=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=len(codes), problem_type="multi_label_classification",
    id2label=dict(enumerate(codes)), label2id={c:i for i,c in enumerate(codes)})
class EvidenceTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            outputs.logits.float(), labels.float(), pos_weight=weights.to(labels.device))
        return (loss, outputs) if return_outputs else loss

trainer = EvidenceTrainer(model=model, args=TrainingArguments(
    output_dir=str(root / "work"), seed=42, num_train_epochs=3,
    learning_rate=2e-5, per_device_train_batch_size=8, per_device_eval_batch_size=16,
    logging_steps=25, logging_first_step=True, disable_tqdm=False, save_strategy="no", report_to="none"), train_dataset=train_features,
    processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer))
print("Bắt đầu huấn luyện 3 epoch.", flush=True)
trainer.train()

In [ ]:
scores = expit(trainer.predict(val_features).predictions)
truth = {r["sample_id"] for r in val if r["overall_label"] == "suspicious"}
gold = {(r["sample_id"], f["code"], sid) for r in val
        for f in r["output"]["findings"] for sid in f["evidence_sentence_ids"]}

def f1(pred, ref):
    return 2 * len(pred & ref) / max(1, len(pred) + len(ref))

def measure(threshold):
    pairs = {(keys[i][0], codes[j], keys[i][1]) for i,j in zip(*np.where(scores >= threshold))}
    detected = {ident for ident, _, _ in pairs}
    return dict(threshold=threshold, missed=len(truth-detected), false_alarms=len(detected-truth),
                label_f1=f1(detected, truth), evidence_f1=f1(pairs, gold))

grid = [measure(i/100) for i in range(5, 100, 5)]
best = max(grid, key=lambda m: (m["label_f1"], m["evidence_f1"], -m["missed"]))
out = root / "result"
checkpoint = out / "checkpoint"
trainer.save_model(str(checkpoint))
tokenizer.save_pretrained(checkpoint)
# Mã tiền xử lý được mang về cùng checkpoint để tái sử dụng khi đánh giá.
(out / "preprocessing.py").write_text("def encode_document(sentences, segmenter, tokenizer):\n    # Giữ câu đang xét ở giữa. Với mẫu dài, ưu tiên phần ngữ cảnh gần câu đó.\n    parts = []\n    for sentence in sentences:\n        text = \" \".join(segmenter.word_segment(sentence[\"text\"]))\n        prefix = sentence[\"type\"] + \" \" + sentence[\"speaker\"] + \": \"\n        parts.append(tokenizer.encode(prefix + text, add_special_tokens=False, verbose=False))\n    sep = [tokenizer.sep_token_id]\n    stream, starts = [], []\n    for part in parts:\n        starts.append(len(stream))\n        stream.extend(part + sep)\n    features = []\n    for start, part in zip(starts, parts):\n        focus = part[:252]\n        available = 252 - len(focus)\n        after = start + len(part) + 1\n        left = min(start, available // 2)\n        right = min(len(stream) - after, available - left)\n        left = min(start, available - right)\n        tokens = stream[start-left:start] + sep + focus + sep + stream[after:after+right]\n        features.append({\"input_ids\": tokenizer.build_inputs_with_special_tokens(tokens)})\n    return features\n", encoding="utf8")
report = dict(model=model_id, codes=codes, seed=42, train_samples=len(train),
              validation_samples=len(val), input_file_sha256=expected_hashes,
              selection="max label F1, then evidence F1, then fewer misses", selected=best, grid=grid)
(out / "validation.json").write_text(json.dumps(report, indent=2), encoding="utf8")
with (out / "validation_scores.jsonl").open("w", encoding="utf8") as handle:
    for (ident, sid), prob in zip(keys, scores, strict=True):
        print(json.dumps(dict(sample_id=ident, sentence_id=sid, scores=dict(zip(codes, prob.tolist())))), file=handle)
archive_path = shutil.make_archive("/kaggle/working/jobshield-phobert-v1-result", "zip", out)
print(best, archive_path)
print("Đã lưu file kết quả:", archive_path)